<a href="https://colab.research.google.com/github/Chathuranga630/ML-Workshop_SLT/blob/main/UAV_Fleet_Simulation_3_Claude.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#!/usr/bin/env python3
"""
Basic UAV fleet coordination simulation
=======================================
A ground control station (GCS, the "Manager") assigns point-visit tasks to a
small fleet of UAVs so that every task is finished before its deadline while
total energy is kept low.  Three attacks can be switched on:

  1. false_battery : UAVs overstate their battery level in status reports
  2. spoofed msgs  : an external attacker forges status messages that
                     impersonate honest UAVs (fake "task done" claims and
                     fake "I am unavailable" claims)
  3. non-coop      : UAVs accept tasks but silently skip them

A `defense` flag enables:
  - HMAC authentication + sequence numbers (rejects forged / replayed msgs)
  - battery cross-check against the GCS's own energy estimate (odometry)
  - task time-outs and heartbeat monitoring
  - a per-UAV trust score; UAVs below the threshold are blacklisted and
    their tasks are reassigned

Usage:
    python uav_fleet_sim.py                 # run all scenarios, print table, save plot
    python uav_fleet_sim.py --seeds 50 --uavs 10
"""
import argparse
import hashlib
import hmac
import json
import math
import random
from dataclasses import dataclass, replace

import numpy as np


# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
@dataclass
class Config:
    n_uavs: int = 8
    n_tasks: int = 24
    area: float = 1000.0          # m, square side
    speed: float = 15.0           # m/s
    dt: float = 1.0               # s
    t_max: float = 400.0          # mission time limit (s)
    battery_j: float = 25000.0    # J per UAV
    e_move: float = 10.0          # J per metre flown
    e_hover: float = 120.0        # J per second hovering (servicing a task)
    service_time: float = 10.0    # s spent at each task location
    reserve_frac: float = 0.10    # battery fraction kept in reserve
    report_period: int = 5        # s between status messages
    seed: int = 0

    # attacks
    n_false_battery: int = 0
    false_battery_bias: float = 0.5   # overstated fraction of capacity
    n_noncoop: int = 0
    p_drop: float = 0.8               # prob. a non-coop UAV skips a task
    spoofer: bool = False
    spoof_period: int = 15            # s between forged messages
    spoof_start: float = 20.0

    # defence
    defense: bool = False
    trust_threshold: float = 0.35
    batt_tol: float = 0.15            # allowed |reported - estimated| / capacity
    batt_penalty: float = 0.15
    timeout_penalty: float = 0.40
    grace: float = 20.0               # s of slack before a task counts as late
    heartbeat_miss: float = 3.0       # missed report periods => UAV declared lost


def dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])


def sign(key, payload):
    msg = json.dumps(payload, sort_keys=True).encode()
    return hmac.new(key, msg, hashlib.sha256).hexdigest()


# ----------------------------------------------------------------------------
# Entities
# ----------------------------------------------------------------------------
@dataclass
class Task:
    id: int
    pos: tuple
    deadline: float
    service: float
    status: str = "pending"      # pending | assigned | done   (manager's belief)
    owner: int = None
    exp_finish: float = 0.0
    truly_done: bool = False     # ground truth
    done_time: float = None
    flagged: bool = False


class UAV:
    def __init__(self, uid, cfg, rng, behavior, base):
        self.id, self.cfg, self.rng, self.behavior = uid, cfg, rng, behavior
        self.base = base
        self.pos = base
        self.battery = cfg.battery_j
        self.queue = []                      # task ids, in execution order
        self.alive = True
        self.seq = 0
        self.key = bytes(rng.getrandbits(8) for _ in range(16))
        self.k = rng.uniform(0.95, 1.05)     # real consumption differs from the nominal model
        self.energy_used = 0.0
        self.pending_done = []
        self.service_tid, self.service_left = None, 0.0
        self.decided = set()

    def step(self, tasks, t):
        cfg = self.cfg
        if not self.alive:
            return
        # a non-cooperative UAV decides, per task, whether to silently skip it
        while self.queue:
            tid = self.queue[0]
            if self.behavior == "noncoop" and tid not in self.decided:
                self.decided.add(tid)
                if self.rng.random() < cfg.p_drop:
                    self.queue.pop(0)
                    continue
            break

        energy = 0.0
        if self.queue:
            task = tasks[self.queue[0]]
            d = dist(self.pos, task.pos)
            if d > 1e-9:
                s = min(cfg.speed * cfg.dt, d)
                self.pos = task.pos if s >= d else tuple(
                    p + (q - p) * s / d for p, q in zip(self.pos, task.pos))
                energy = cfg.e_move * s
            else:
                if self.service_tid != task.id:
                    self.service_tid, self.service_left = task.id, task.service
                energy = cfg.e_hover * cfg.dt
                self.service_left -= cfg.dt
                if self.service_left <= 0:
                    task.truly_done, task.done_time = True, t + cfg.dt
                    self.pending_done.append(task.id)
                    self.queue.pop(0)
                    self.service_tid = None
        else:                                 # idle -> fly home, then sit at base
            d = dist(self.pos, self.base)
            if d > 1e-9:
                s = min(cfg.speed * cfg.dt, d)
                self.pos = self.base if s >= d else tuple(
                    p + (q - p) * s / d for p, q in zip(self.pos, self.base))
                energy = cfg.e_move * s

        energy *= self.k
        self.battery -= energy
        self.energy_used += energy
        if self.battery <= 0:
            self.battery, self.alive = 0.0, False   # crashed / out of power

    def status(self, t):
        if not self.alive:
            return None
        self.seq += 1
        rep = self.battery
        if self.behavior == "false_battery":
            rep = min(self.cfg.battery_j,
                      self.battery + self.cfg.false_battery_bias * self.cfg.battery_j)
        payload = dict(type="status", uid=self.id, seq=self.seq, t=t,
                       pos=list(self.pos), batt=rep, done=list(self.pending_done))
        self.pending_done.clear()
        return dict(payload=payload, tag=sign(self.key, payload))


class Belief:
    """What the GCS currently thinks about one UAV."""
    def __init__(self, cfg, base):
        self.pos, self.last_pos = base, base
        self.batt_rep = cfg.battery_j
        self.est = cfg.battery_j           # GCS's own energy estimate
        self.last_seen, self.last_seq = 0.0, 0
        self.trust = 1.0
        self.blacklisted = False
        self.unavailable = False
        self.detect_time = None


# ----------------------------------------------------------------------------
# Ground control station
# ----------------------------------------------------------------------------
class Manager:
    def __init__(self, cfg, uavs, tasks, base):
        self.cfg, self.uavs, self.tasks, self.base = cfg, uavs, tasks, base
        self.b = {u.id: Belief(cfg, base) for u in uavs}
        self.queue = {u.id: [] for u in uavs}
        self.spoof_rejected = 0
        self.spoof_accepted = 0

    # ---- planning helpers -------------------------------------------------
    def project(self, uid, t):
        """End position/time/energy if the UAV executes its queue from its believed position."""
        cfg = self.cfg
        pos, time, energy = self.b[uid].pos, t, 0.0
        for tid in self.queue[uid]:
            task = self.tasks[tid]
            d = dist(pos, task.pos)
            time += d / cfg.speed + task.service
            energy += cfg.e_move * d + cfg.e_hover * task.service
            pos = task.pos
        return pos, time, energy

    def avail_batt(self, uid):
        b = self.b[uid]
        return min(b.batt_rep, b.est) if self.cfg.defense else b.batt_rep

    def usable(self, uid):
        b = self.b[uid]
        return not (b.blacklisted or b.unavailable)

    # ---- task bookkeeping ---------------------------------------------------
    def assign(self, task, uid, exp_finish):
        task.status, task.owner, task.exp_finish = "assigned", uid, exp_finish
        self.queue[uid].append(task.id)
        self.uavs[uid].queue.append(task.id)        # trusted command channel

    def release(self, tid):
        task = self.tasks[tid]
        if task.owner is not None:
            if tid in self.queue[task.owner]:
                self.queue[task.owner].remove(tid)
            if tid in self.uavs[task.owner].queue:
                self.uavs[task.owner].queue.remove(tid)
        task.owner = None

    def revoke_all(self, uid):
        for tid in list(self.queue[uid]):
            self.release(tid)
            if self.tasks[tid].status != "done":
                self.tasks[tid].status = "pending"
                self.tasks[tid].flagged = False

    def mark_done(self, tid):
        self.release(tid)
        self.tasks[tid].status = "done"

    # ---- message handling -----------------------------------------------------
    def receive(self, msg, t, forged=False):
        cfg = self.cfg
        p, tag = msg["payload"], msg["tag"]
        uid = p["uid"]
        b = self.b[uid]

        if cfg.defense:
            if not hmac.compare_digest(tag, sign(self.uavs[uid].key, p)) or p["seq"] <= b.last_seq:
                self.spoof_rejected += 1
                return
        if forged:
            self.spoof_accepted += 1
        b.last_seq = max(b.last_seq, p["seq"])

        if p["type"] == "unavailable":
            self.revoke_all(uid)
            b.unavailable = True
            return

        # ordinary status message
        dt_rep = max(t - b.last_seen, 1e-9)
        d = dist(b.pos, p["pos"])
        airborne = dist(b.pos, self.base) > 1e-6 or dist(p["pos"], self.base) > 1e-6
        hover = max(0.0, dt_rep - d / cfg.speed) if airborne else 0.0
        b.est -= cfg.e_move * d + cfg.e_hover * hover
        b.pos, b.batt_rep, b.last_seen = tuple(p["pos"]), p["batt"], t

        for tid in p["done"]:
            if cfg.defense and tid not in self.queue[uid]:
                continue                       # claim for a task we never gave this UAV
            self.mark_done(tid)

        if cfg.defense and abs(p["batt"] - b.est) / cfg.battery_j > cfg.batt_tol:
            self.penalise(uid, cfg.batt_penalty, t)

    def penalise(self, uid, amount, t):
        b = self.b[uid]
        b.trust = max(0.0, b.trust - amount)
        if b.trust < self.cfg.trust_threshold and not b.blacklisted:
            b.blacklisted, b.detect_time = True, t
            self.revoke_all(uid)

    # ---- monitoring (defence only) ---------------------------------------------
    def monitor(self, t):
        cfg = self.cfg
        for u in self.uavs:
            b = self.b[u.id]
            if b.blacklisted:
                continue
            if t - b.last_seen > cfg.heartbeat_miss * cfg.report_period + 2:
                b.blacklisted, b.detect_time = True, t     # UAV went silent
                self.revoke_all(u.id)
                continue
            for tid in list(self.queue[u.id]):
                task = self.tasks[tid]
                if t > task.exp_finish + cfg.grace and not task.flagged:
                    task.flagged = True
                    self.release(tid)
                    task.status = "pending"
                    self.penalise(u.id, cfg.timeout_penalty, t)

    # ---- allocation ----------------------------------------------------------------
    def allocate(self, t):
        """Greedy: tasks by deadline; pick the feasible UAV with the lowest
        trust-weighted marginal energy cost."""
        cfg = self.cfg
        pending = sorted((x for x in self.tasks if x.status == "pending"),
                         key=lambda x: x.deadline)
        for task in pending:
            best, best_cost, best_fin = None, float("inf"), 0.0
            for u in self.uavs:
                if not self.usable(u.id):
                    continue
                end_pos, end_time, q_energy = self.project(u.id, t)
                d = dist(end_pos, task.pos)
                finish = end_time + d / cfg.speed + task.service
                if finish > task.deadline:
                    continue
                e_task = cfg.e_move * d + cfg.e_hover * task.service
                e_back = cfg.e_move * dist(task.pos, self.base)
                left = self.avail_batt(u.id) - q_energy - e_task - e_back
                if left < cfg.reserve_frac * cfg.battery_j:
                    continue
                cost = e_task * (1 + 2 * (1 - self.b[u.id].trust))
                if cost < best_cost:
                    best, best_cost, best_fin = u.id, cost, finish
            if best is not None:
                self.assign(task, best, best_fin)


# ----------------------------------------------------------------------------
# Attacker
# ----------------------------------------------------------------------------
def spoofed_messages(cfg, mgr, rng, t):
    """External attacker forges messages without knowing any UAV's key."""
    if not cfg.spoofer or t < cfg.spoof_start or int(t) % cfg.spoof_period != 0:
        return []
    victim = rng.choice(mgr.uavs).id
    b = mgr.b[victim]
    payload = dict(type="status", uid=victim, seq=b.last_seq + 1, t=t,
                   pos=list(b.pos), batt=b.batt_rep, done=[])
    if rng.random() < 0.5:                              # fake "task done"
        open_tasks = [x.id for x in mgr.tasks if not x.truly_done and x.status != "done"]
        if not open_tasks:
            return []
        payload["done"] = [rng.choice(open_tasks)]
    else:                                               # fake "I'm unavailable"
        payload["type"] = "unavailable"
    return [dict(payload=payload, tag="%064x" % rng.getrandbits(256))]


# ----------------------------------------------------------------------------
# Simulation
# ----------------------------------------------------------------------------
def run(cfg: Config):
    rng = random.Random(cfg.seed)
    base = (cfg.area / 2, cfg.area / 2)
    tasks = [Task(i, (rng.uniform(0, cfg.area), rng.uniform(0, cfg.area)),
                  rng.uniform(0.35 * cfg.t_max, cfg.t_max), cfg.service_time)
             for i in range(cfg.n_tasks)]

    ids = list(range(cfg.n_uavs))
    rng.shuffle(ids)
    behavior = {i: "honest" for i in ids}
    bad_fb = ids[:cfg.n_false_battery]
    bad_nc = ids[cfg.n_false_battery:cfg.n_false_battery + cfg.n_noncoop]
    for i in bad_fb:
        behavior[i] = "false_battery"
    for i in bad_nc:
        behavior[i] = "noncoop"
    malicious = set(bad_fb) | set(bad_nc)

    uavs = [UAV(i, cfg, rng, behavior[i], base) for i in range(cfg.n_uavs)]
    mgr = Manager(cfg, uavs, tasks, base)
    atk_rng = random.Random(cfg.seed + 999)

    steps = int(cfg.t_max / cfg.dt)
    for k in range(steps):
        t = k * cfg.dt
        if k % cfg.report_period == 0:
            for u in uavs:
                m = u.status(t)
                if m:
                    mgr.receive(m, t)
        for m in spoofed_messages(cfg, mgr, atk_rng, t):
            mgr.receive(m, t, forged=True)
        if cfg.defense:
            mgr.monitor(t)
        mgr.allocate(t)
        for u in uavs:
            u.step(tasks, t)

    done = [x for x in tasks if x.truly_done]
    on_time = [x for x in done if x.done_time <= x.deadline]
    energy = sum(u.energy_used for u in uavs)
    flagged = {i for i, b in mgr.b.items() if b.blacklisted}
    det_times = [mgr.b[i].detect_time for i in malicious if mgr.b[i].blacklisted]
    return dict(
        completed=len(done) / cfg.n_tasks,
        on_time=len(on_time) / cfg.n_tasks,
        believed=sum(x.status == "done" for x in tasks) / cfg.n_tasks,
        energy_kj=energy / 1000,
        kj_per_task=(energy / 1000) / max(len(done), 1),
        crashed=sum(not u.alive for u in uavs),
        tp=len(flagged & malicious), n_mal=len(malicious),
        fp=len(flagged - malicious),
        detect_s=float(np.mean(det_times)) if det_times else float("nan"),
        spoof_ok=mgr.spoof_accepted, spoof_rej=mgr.spoof_rejected,
    )


def average(cfg, seeds):
    runs = [run(replace(cfg, seed=s)) for s in range(seeds)]
    return {k: float(np.nanmean([r[k] for r in runs])) if any(
        not math.isnan(r[k]) for r in runs) else float("nan") for k in runs[0]}


SCENARIOS = {
    "baseline (no attack)": dict(),
    "false battery (2 UAVs)": dict(n_false_battery=2),
    "non-cooperative (2 UAVs)": dict(n_noncoop=2),
    "spoofed messages": dict(spoofer=True),
    "all attacks combined": dict(n_false_battery=2, n_noncoop=2, spoofer=True),
}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--uavs", type=int, default=8)
    ap.add_argument("--tasks", type=int, default=24)
    ap.add_argument("--seeds", type=int, default=30)
    ap.add_argument("--plot", default="uav_results.png")
    # Use an empty list inside interactive sessions like Google Colab
    # to prevent argparse from checking Jupyter connection parameters
    a = ap.parse_args([])
    base_cfg = Config(n_uavs=a.uavs, n_tasks=a.tasks)

    hdr = (f"{'scenario':26s} {'def':>3s} {'done%':>6s} {'ontime%':>7s} {'belief%':>7s} "
           f"{'kJ':>6s} {'kJ/task':>7s} {'crash':>5s} {'TP':>5s} {'FP':>4s} "
           f"{'t_det':>6s} {'spoofOK':>7s} {'spoofRej':>8s}")
    print(f"{a.uavs} UAVs, {a.tasks} tasks, averaged over {a.seeds} seeds\n")
    print(hdr)
    print("-" * len(hdr))
    results = {}
    for name, kw in SCENARIOS.items():
        for d in (False, True):
            r = average(replace(base_cfg, defense=d, **kw), a.seeds)
            results[(name, d)] = r
            tp = f"{r['tp']:.1f}/{r['n_mal']:.0f}" if r["n_mal"] else "-"
            print(f"{name:26s} {'on' if d else 'off':>3s} {100*r['completed']:6.1f} "
                  f"{100*r['on_time']:7.1f} {100*r['believed']:7.1f} {r['energy_kj']:6.0f} "
                  f"{r['kj_per_task']:7.2f} {r['crashed']:5.1f} {tp:>5s} {r['fp']:4.1f} "
                  f"{r['detect_s']:6.0f} {r['spoof_ok']:7.1f} {r['spoof_rej']:8.1f}")
        print()

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        names = list(SCENARIOS)
        x = np.arange(len(names))
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
        for j, (key, title) in enumerate([("completed", "Tasks truly completed (%)"),
                                           ("kj_per_task", "Energy per completed task (kJ)")]):
            scale = 100 if key == "completed" else 1
            ax[j].bar(x - 0.2, [scale * results[(n, False)][key] for n in names], 0.4,
                      label="no defence")
            ax[j].bar(x + 0.2, [scale * results[(n, True)][key] for n in names], 0.4,
                      label="with defence")
            ax[j].set_xticks(x)
            ax[j].set_xticklabels([n.replace(" (", "\n(") for n in names], fontsize=8)
            ax[j].set_title(title)
            ax[j].legend()
        plt.tight_layout()
        plt.savefig(a.plot, dpi=140)
        print(f"plot saved to {a.plot}")
    except ImportError:
        pass


if __name__ == "__main__":
    main()


8 UAVs, 24 tasks, averaged over 30 seeds

scenario                   def  done% ontime% belief%     kJ kJ/task crash    TP   FP  t_det spoofOK spoofRej
-------------------------------------------------------------------------------------------------------------
baseline (no attack)       off  100.0   100.0   100.0    118    4.91   0.0     -  0.0    nan     0.0      0.0
baseline (no attack)        on  100.0   100.0   100.0    118    4.91   0.0     -  0.0    nan     0.0      0.0

false battery (2 UAVs)     off  100.0   100.0   100.0    118    4.91   0.0 0.0/2  0.0    nan     0.0      0.0
false battery (2 UAVs)      on   98.9    98.9    98.9    128    5.39   0.0 1.9/2  0.0     52     0.0      0.0

non-cooperative (2 UAVs)   off   78.8    78.8    78.8     99    5.24   0.0 0.0/2  0.0    nan     0.0      0.0
non-cooperative (2 UAVs)    on   96.7    96.7    96.7    122    5.26   0.0 1.6/2  0.0     82     0.0      0.0

spoofed messages           off   84.7    84.7   100.0    118    5.84   0.0 